# Load a 3D NIfTI with itkwidgets (JupyterLite)

This notebook runs in the **Pyodide** kernel. Dependencies come from the docs checked into `PythonLightSuite/docs/`:

- **LightSuite** (`docs/LightSuite/`): atlas pipelines use NIfTI files; the Python helper `scripts/lsfm_atlas_inspect` loads volumes with **nibabel**, **numpy**, **matplotlib**, and **pandas**.
- **ITK-Wasm** (`docs/ITK-Wasm-main/`): WASM-friendly I/O via **itkwasm** / **itkwasm-image-io**. On Pyodide/emscripten only async readers exist — use **`await imread_async(...)`**, not `imread`.
- **itkwidgets** (`docs/itkwidgets-main`): interactive web viewers via `itkwidgets.view`; in JupyterLite use **`piplite`** and install **`imjoy-jupyterlab-extension`** in the site build (see [itkwidgets deployments](https://itkwidgets.readthedocs.io/en/latest/deployments.html)). **`ngff-zarr`** (used by `view()`) expects **Zarr 3** for OME-Zarr ≥0.5, while itkwidgets still declares **`zarr<3`**. Micropip **merges** `constraints=` with that pin into **`zarr<3,>=3`** (unsatisfiable). The install cell therefore does **two steps**: resolve itkwidgets (gets Zarr 2), then **`reinstall=True`** to upgrade Zarr to 3.x.

Below we fetch a small public `.nii`, read it with `await imread_async(...)`, and display it with itkwidgets.


In [ ]:
import piplite

# ngff-zarr (itkwidgets -> view()) needs zarr-python 3.x; itkwidgets metadata pins zarr<3.
# Do NOT use constraints=["zarr>=3"] together with itkwidgets — micropip merges to zarr<3 AND >=3 (impossible).
await piplite.install(
    [
        "itkwidgets>=1.0a55",
        "itkwasm-image-io",
    ],
)
await piplite.install(
    ["zarr>=3.0.0b2"],
    reinstall=True,
)


In [ ]:
from pyodide.http import pyfetch

# Small test volume from nibabel (BSD); suitable for smoke-testing I/O + viewer
NIFTI_URL = (
    "https://raw.githubusercontent.com/nipy/nibabel/master/"
    "nibabel/tests/data/anatomical.nii"
)


async def download(url: str, path: str) -> int:
    resp = await pyfetch(url)
    data = await resp.bytes()
    with open(path, "wb") as f:
        f.write(data)
    return len(data)


local_path = "anatomical.nii"
nbytes = await download(NIFTI_URL, local_path)
print(f"Downloaded {local_path} ({nbytes} bytes)")


In [ ]:
from itkwasm_image_io import imread_async
from itkwidgets import view

image = await imread_async(local_path)
view(image, rotate=True)
